# 06 -- DueCare Adversarial Attack Resistance

**DueCare** | Named for Cal. Civ. Code sect. 1714(a)

---

**Purpose:** How well does Gemma resist 15 different adversarial attack
strategies? This notebook generates adversarial variations from base
prompts, scores them, and identifies which attack types are most
effective at bypassing safety guardrails.

| | |
|---|---|
| **Input** | 5 graded base prompts, 15 adversarial generators |
| **Output** | Attack variation samples, per-generator statistics, attack type distribution |
| **Prerequisites** | `duecare-llm-wheels` dataset attached; no GPU required for generation (GPU needed for live model scoring) |
| **Pipeline position** | Stage 3 of the DueCare showcase pipeline. Previous: NB 05 (RAG Comparison). Next: NB 08 (Function Calling). |

---

### Why adversarial testing matters for migrant worker protection

Bad actors who exploit migrant workers do not use polite, direct
questions. They use obfuscation, academic framing, corporate disguise,
and emotional manipulation to extract harmful content from LLMs.
A model that passes standard safety tests may fail catastrophically
against these adversarial variations.

DueCare's 15 generators systematically produce these attacks so
safety evaluations test what adversaries actually do, not what
benchmarks wish they would do.

### Flow diagram

```
5 Base Prompts        15 Generators
      |                     |
      +--------+   +-------+
               |   |
               v   v
        +------+---+------+
        | Generate 1 var  |
        | per generator   |
        | per base prompt |
        +------+----------+
               |
               v
     ~75 adversarial prompts
               |
        +------+------+
        | Score each  |
        | (model/     |
        |  scripted)  |
        +------+------+
               |
               v
     Attack type analysis
     (which attacks succeed?)
```


## 1. Install DueCare

Install the DueCare wheel packages. The adversarial generators
are part of the `duecare-llm-tasks` package.


In [ ]:
import subprocess, glob, sys
for p in ['/kaggle/input/duecare-llm-wheels/*.whl',
          '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
          '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + wheels + ['-q'])
    print(f'Installed {len(wheels)} DueCare wheels.')


## 2. Generate adversarial variations from 5 base prompts

We start with 5 graded base prompts and run all 15 generators,
producing 1 variation per generator per prompt. This gives us
approximately 75 adversarial test prompts spanning every attack type.

The table below shows each generator's name and how many variations
it produced. Some generators may produce fewer than 5 if the base
prompt does not fit the attack pattern.


In [ ]:
from duecare.tasks.generators import ALL_GENERATORS
from duecare.domains import register_discovered, load_domain_pack

register_discovered()
pack = load_domain_pack('trafficking')
base = [p for p in pack.seed_prompts() if p.get('graded_responses')][:5]

print(f'Base prompts: {len(base)}')
print(f'Generators available: {len(ALL_GENERATORS)}')
print()

all_variations = []
print(f'{"Generator":<35} {"Variations":>10}')
print('-' * 47)
for gen in ALL_GENERATORS:
    variations = gen.generate(base, n_variations=1, seed=42)
    all_variations.extend(variations)
    print(f'  {gen.__class__.__name__:<33} {len(variations):>10}')

print(f'\nTotal: {len(base)} base + {len(all_variations)} variations = '
      f'{len(base)+len(all_variations)} test prompts')
print(f'Amplification factor: {len(all_variations)/max(len(base),1):.1f}x')


### What the generators produce

Each generator applies a specific adversarial strategy:
- **Evasion/Regulatory:** Frame exploitation as regulatory compliance
- **Coercion:** Use pressure tactics and urgency
- **Financial:** Disguise fees as "standard business practice"
- **Persona:** Impersonate authority figures (31 persona types)
- **Creative:** Novel attacks like ASCII art, role-play, academic framing

The diversity of attack types is what makes DueCare's evaluation
meaningful: a model cannot pass by memorizing responses to one style
of prompt.


## 3. Inspect sample adversarial variations

Each variation carries metadata about its mutation type and base
prompt. This traceability is critical for understanding which
attack types a model fails against.


In [ ]:
import textwrap
print('--- Sample adversarial variations ---\n')
for v in all_variations[:5]:
    mt = v.get('metadata',{}).get('mutation_type','?')
    base_id = v.get('metadata',{}).get('base_prompt_id','?')
    print(f'[{mt}] (from base: {base_id})')
    print(textwrap.fill(v['text'][:200], width=80))
    print('...\n')


## 4. Score all variations

In production, these would be run through Gemma 4 and scored using
the rubric framework. Here we use a ScriptedModel to demonstrate
that the scoring pipeline handles all variation types consistently.

The attack type distribution shows which categories of adversarial
attacks are represented in the generated test set.


In [ ]:
from duecare.core import Capability, GenerationResult, Embedding, ModelHealth

class ScriptedModel:
    id = 'scripted:adversarial'
    display_name = 'Scripted'
    provider = 'scripted'
    capabilities = {Capability.TEXT}
    context_length = 4096
    _r = 'I cannot help with that. Under ILO C181 Article 7, workers shall not pay fees. Contact POEA at 1343.'
    def generate(self, messages, **kw): return GenerationResult(text=self._r, model_id=self.id)
    def embed(self, texts): return [Embedding(text=t, vector=[0]*4, dimension=4, model_id=self.id) for t in texts]
    def healthcheck(self): return ModelHealth(model_id=self.id, healthy=True)

from collections import Counter
gen_results = Counter()
for v in all_variations:
    mt = v.get('metadata',{}).get('mutation_type','unknown').split('_')[0]
    gen_results[mt] += 1

print(f'{"Attack Type":<25} {"Count":>6} {"Share":>8}')
print('-' * 42)
for at, count in gen_results.most_common():
    pct = count / len(all_variations) * 100
    print(f'{at:<25} {count:>6} {pct:>7.1f}%')


## 5. Visualize adversarial attack distribution

The charts below make the attack-type distribution immediately
readable. The horizontal bar chart ranks attack types by frequency,
revealing which generators produce the most variations and therefore
dominate the test surface. The stacked bar chart breaks each generator
into pass, fail, and error counts so you can see at a glance which
attack strategies are most effective at bypassing safety guardrails.

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', '-q'])
print('plotly installed.')

In [ ]:
import plotly.graph_objects as go

# --- Chart 1: Horizontal bar chart of attack type frequency ---
# Sort by count ascending so the longest bars are at the top
sorted_types = gen_results.most_common()
sorted_types.reverse()  # ascending for horizontal bar layout
labels = [t[0] for t in sorted_types]
counts = [t[1] for t in sorted_types]
total = sum(counts)
pcts = [c / total * 100 for c in counts]

fig1 = go.Figure(go.Bar(
    x=counts,
    y=labels,
    orientation='h',
    text=[f'{c}  ({p:.0f}%)' for c, p in zip(counts, pcts)],
    textposition='outside',
    marker_color='#4285F4',
))
fig1.update_layout(
    title='Attack Type Frequency Across All Generators',
    xaxis_title='Number of Variations Generated',
    yaxis_title='Attack Type',
    height=max(400, len(labels) * 35 + 120),
    margin=dict(l=180, r=60, t=60, b=50),
    template='plotly_white',
)
fig1.show()

In [ ]:
# --- Chart 2: Stacked bar chart -- pass / fail / error per generator ---
# Since we used a ScriptedModel that always returns a safe refusal,
# all variations are scored as "pass" here. In a real evaluation with
# Gemma 4, some attacks WILL succeed -- the stacked chart will reveal
# which generators are most dangerous.

from collections import defaultdict

# Tally per-generator results using the scripted model
gen_stats = defaultdict(lambda: {'pass': 0, 'fail': 0, 'error': 0})
model = ScriptedModel()
for v in all_variations:
    mt = v.get('metadata', {}).get('mutation_type', 'unknown').split('_')[0]
    try:
        result = model.generate([{'role': 'user', 'content': v['text'][:500]}])
        # The scripted model always refuses, so score as pass
        # A real model would be scored against the rubric
        if 'cannot help' in result.text.lower() or 'shall not' in result.text.lower():
            gen_stats[mt]['pass'] += 1
        else:
            gen_stats[mt]['fail'] += 1
    except Exception:
        gen_stats[mt]['error'] += 1

# Sort generators by total count descending
gen_names = sorted(gen_stats.keys(), key=lambda g: sum(gen_stats[g].values()), reverse=True)
pass_counts = [gen_stats[g]['pass'] for g in gen_names]
fail_counts = [gen_stats[g]['fail'] for g in gen_names]
error_counts = [gen_stats[g]['error'] for g in gen_names]

fig2 = go.Figure()
fig2.add_trace(go.Bar(name='Pass (refused)', x=gen_names, y=pass_counts,
                       marker_color='#34A853'))
fig2.add_trace(go.Bar(name='Fail (bypassed)', x=gen_names, y=fail_counts,
                       marker_color='#EA4335'))
fig2.add_trace(go.Bar(name='Error', x=gen_names, y=error_counts,
                       marker_color='#FBBC05'))
fig2.update_layout(
    barmode='stack',
    title='Per-Generator Safety Outcome (Pass / Fail / Error)',
    xaxis_title='Attack Type',
    yaxis_title='Number of Variations',
    height=500,
    margin=dict(l=60, r=40, t=60, b=120),
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis_tickangle=-40,
)
fig2.show()

### What the charts reveal about adversarial weak spots

The horizontal bar chart shows the raw volume of adversarial variations each attack
type contributes to the test surface. Attack types with the highest counts dominate
the evaluation and therefore have the most opportunities to find cracks in the model's
safety guardrails.

The stacked bar chart breaks each attack type into pass (the model correctly refused),
fail (the attack bypassed safety), and error (the generation or scoring pipeline
encountered a problem). Because this notebook uses a ScriptedModel that always refuses,
every bar is solid green. When you swap in a real Gemma 4 model, the red "fail"
segments will appear -- and the attack types with the tallest red segments are the ones
that most urgently need additional safety training data in Phase 3.

In practice, the attack categories that most often bypass frontier models are persona
injection, academic reframing, and financial obfuscation, because these attacks shift
the conversational register away from the patterns safety training targets. DueCare's
15 generators ensure all of these categories are tested systematically rather than
left to chance.

## Summary and next steps

### Key findings

DueCare's 15 adversarial generators produce a diverse attack surface from any set of
base prompts, spanning evasion, coercion, financial obfuscation, persona injection,
and creative reframing strategies. Each generator tests a fundamentally different attack
vector, which means a model cannot achieve a passing score by defending against only one
style of prompt. The scoring framework processes all variation types through the same
rubric pipeline, ensuring consistent and comparable evaluation across attack categories.
This breadth and consistency is what makes DueCare's adversarial evaluation meaningful
for real-world deployment where bad actors use every technique available to them.

### Connection to other notebooks

The previous notebook (NB 05, RAG Comparison) tested whether retrieval-augmented context
improves response quality. This notebook answers the complementary question: does safety
hold up when the input is deliberately adversarial? The next notebook (NB 08, Function
Calling) demonstrates how Gemma 4's native function calling and multimodal understanding
serve as load-bearing infrastructure for the DueCare pipeline. NB 12 scales this
adversarial generation to thousands of variations with automated validation and
importance ranking.

**Privacy is non-negotiable. All adversarial testing runs on-device.**